# FakeProductDetector — MobileNetV3 Transfer Learning

This notebook trains a binary image classifier to distinguish **authentic** products from **fake/counterfeit** ones.
The trained model is exported to TFLite (INT8 quantized) for on-device inference in the Android app.

**Pipeline overview:**
1. Mount Google Drive (optional) and load the labelled dataset.
2. Apply data augmentation to reduce overfitting on small datasets.
3. **Phase 1 — Frozen base:** Train only the classification head on top of MobileNetV3Small (ImageNet weights).
4. **Phase 2 — Fine-tuning:** Unfreeze the top layers of the base model and train end-to-end at a low learning rate.
5. Evaluate the model: accuracy, precision, recall, F1, confusion matrix.
6. Convert to INT8-quantized TFLite and copy to the Android `assets/` folder.

> **Dataset structure expected:**
> ```
> dataset/
>   train/
>     authentic/   ← images of genuine products
>     fake/        ← images of counterfeit products
>   validation/
>     authentic/
>     fake/
> ```

## 1. Environment Setup

Install required packages (safe to re-run; pip will skip already-installed packages).
Then import all libraries and verify the TensorFlow version and GPU availability.

In [ ]:
# Install required packages
!pip install -q tensorflow scikit-learn matplotlib

import os
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Print TensorFlow version so results are reproducible
print(f"TensorFlow version: {tf.__version__}")

# List available GPUs; an empty list means training will use CPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU available: {gpus}")
    # Enable memory growth to avoid allocating all GPU memory at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPU found — training will run on CPU (slower).")
    print("Consider using Google Colab with a T4 runtime for faster training.")

## 2. Mount Google Drive (optional — skip if running locally)

If running in Google Colab, mount your Drive so you can access a dataset stored there.
The cell is safe to run in any environment — it catches the `ImportError` that occurs outside Colab.

In [ ]:
# Attempt to mount Google Drive (only works inside Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted at /content/drive")
except ImportError:
    # Not running in Colab — Drive is not available
    print("Not running in Colab — skipping Drive mount.")
    print("Dataset should be available locally at the path set in the next cell.")

## 3. Dataset Configuration

Define all hyper-parameters and paths in one place so they are easy to adjust.

- **`EPOCHS_FROZEN`** — epochs for Phase 1 (only the head trains).
- **`EPOCHS_FINETUNE`** — additional epochs for Phase 2 (top layers unfrozen).
- **`FINE_TUNE_AT`** — number of layers to unfreeze from the end of the base model.
- **`LEARNING_RATE_FINETUNE`** — must be much smaller than Phase 1 to avoid destroying pre-trained weights.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust this path if the dataset lives on Google Drive, e.g.:
# DATASET_DIR = Path("/content/drive/MyDrive/FakeProductDetector/dataset")
DATASET_DIR = Path("dataset")

# ── Image / batch settings ───────────────────────────────────────────────────
IMG_SIZE    = (224, 224)   # MobileNetV3Small default input resolution
BATCH_SIZE  = 32           # Reduce to 16 if you hit OOM errors on GPU

# ── Training schedule ────────────────────────────────────────────────────────
EPOCHS_FROZEN    = 20      # Phase 1: train head only
EPOCHS_FINETUNE  = 10      # Phase 2: fine-tune top layers
FINE_TUNE_AT     = 30      # Unfreeze the last N layers of the base model

# ── Learning rates ───────────────────────────────────────────────────────────
LEARNING_RATE_FROZEN   = 1e-3   # Standard Adam LR for head training
LEARNING_RATE_FINETUNE = 1e-5   # Very small LR to preserve pre-trained features

# ── TF dataset optimisation ──────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE   # Let TF decide parallelism at runtime

# ── Class labels (must match directory names inside train/ and validation/) ──
CLASS_NAMES = ["authentic", "fake"]   # index 0 = authentic, index 1 = fake

# ── Verify the dataset directory exists ──────────────────────────────────────
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATASET_DIR.resolve()}\n"
        "Please create the folder structure described in the title cell."
    )

print(f"Dataset root : {DATASET_DIR.resolve()}")
print()

# Count images per split and class to verify the dataset is balanced
for split in ["train", "validation"]:
    split_dir = DATASET_DIR / split
    print(f"  {split}/")
    total = 0
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if cls_dir.exists():
            # Count only common image extensions
            n = len([
                f for f in cls_dir.iterdir()
                if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            ])
            print(f"    {cls:12s}: {n:>5} images")
            total += n
        else:
            print(f"    {cls:12s}: ⚠  directory missing ({cls_dir})")
    print(f"    {'TOTAL':12s}: {total:>5} images")
    print()

## 4. Data Loading and Augmentation

Images are loaded directly from the directory structure using `image_dataset_from_directory`.
Augmentation is applied **only to the training set** to artificially increase dataset diversity.
MobileNetV3 requires its own preprocessing (pixel values scaled to `[-1, 1]`).

In [ ]:
# ── Load raw datasets from disk ───────────────────────────────────────────────
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',      # returns float32 labels (0.0 or 1.0)
    class_names=CLASS_NAMES,  # enforce ordering: authentic=0, fake=1
    seed=42,
    shuffle=True,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    class_names=CLASS_NAMES,
    seed=42,
    shuffle=False,            # keep validation order deterministic
)

print(f"Class names (as loaded): {train_ds_raw.class_names}")

# ── Data augmentation layer (training only) ───────────────────────────────────
# These transformations are applied randomly during each training step.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),    # mirror horizontally
    tf.keras.layers.RandomRotation(0.1),          # ±36° rotation
    tf.keras.layers.RandomZoom(0.1),              # ±10% zoom
    tf.keras.layers.RandomBrightness(0.1),        # ±10% brightness shift
], name="augmentation")

# MobileNetV3-specific preprocessing: scales [0, 255] → [-1, 1]
preprocess = tf.keras.layers.Lambda(
    tf.keras.applications.mobilenet_v3.preprocess_input,
    name="mobilenetv3_preprocess"
)

# ── Apply augmentation + preprocessing and optimise pipeline ─────────────────
def prepare_train(image, label):
    """Augment and preprocess a training batch."""
    image = augmentation(image, training=True)  # only augment during training
    image = tf.keras.applications.mobilenet_v3.preprocess_input(image)
    return image, label

def prepare_val(image, label):
    """Preprocess a validation batch (no augmentation)."""
    image = tf.keras.applications.mobilenet_v3.preprocess_input(image)
    return image, label

# Cache → map → prefetch is the recommended TF performance pattern
train_ds = (
    train_ds_raw
    .cache()                          # cache raw images in memory after first epoch
    .map(prepare_train, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)               # overlap data loading with GPU computation
)

val_ds = (
    val_ds_raw
    .cache()
    .map(prepare_val, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

# Count batches to report approximate image totals
n_train_batches = sum(1 for _ in train_ds)
n_val_batches   = sum(1 for _ in val_ds)
print(f"\nTraining batches  : {n_train_batches}  (~{n_train_batches * BATCH_SIZE} images)")
print(f"Validation batches: {n_val_batches}  (~{n_val_batches * BATCH_SIZE} images)")

## 5. Visualise Sample Training Images

A quick sanity-check: plot a 3×3 grid of training images with their ground-truth labels.
Images are shown **before** MobileNetV3 preprocessing (using the raw dataset) so colours look natural.

In [ ]:
# Take one batch from the *raw* (unprocessed) training dataset for visualisation
sample_images, sample_labels = next(iter(train_ds_raw))

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
fig.suptitle("Sample Training Images", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    # Convert tensor to uint8 for display
    img = sample_images[i].numpy().astype("uint8")
    # label is a float32 scalar (0.0 or 1.0) — map to class name
    label_idx = int(sample_labels[i].numpy())
    label_name = CLASS_NAMES[label_idx]

    ax.imshow(img)
    ax.set_title(f"{label_name}", color='green' if label_idx == 0 else 'red', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig("sample_training_images.png", dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved as sample_training_images.png")

## 6. Build the Model — Frozen Base (Phase 1)

We use **MobileNetV3Small** pre-trained on ImageNet as a feature extractor.
In Phase 1 the base model weights are **frozen** — only the classification head (Dense layers) trains.
This is fast and avoids overfitting when the dataset is small.

Architecture:
```
Input (224×224×3)
  └─ Augmentation (train-time only)
       └─ MobileNetV3Small (frozen)
            └─ GlobalAveragePooling2D
                 └─ Dense(128, relu)
                      └─ Dropout(0.3)
                           └─ Dense(1, sigmoid)  → P(fake)
```

In [ ]:
# ── Load MobileNetV3Small base model ─────────────────────────────────────────
base_model = tf.keras.applications.MobileNetV3Small(
    weights='imagenet',           # start from ImageNet pre-trained weights
    include_top=False,            # exclude the original classification head
    input_shape=(224, 224, 3),    # match our image size
)

# Freeze all base model layers — their weights will NOT update during Phase 1
base_model.trainable = False

# ── Build model using the Functional API ─────────────────────────────────────
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")

# Pass through the augmentation Sequential model (active only during training)
x = augmentation(inputs)

# Apply MobileNetV3 preprocessing: pixel values [0,255] → [-1, 1]
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

# Run the frozen feature extractor; training=False disables BatchNorm updates
x = base_model(x, training=False)

# Collapse spatial dimensions to a single feature vector per image
x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)

# Fully-connected classification head
x = tf.keras.layers.Dense(128, activation='relu', name="dense_head")(x)
x = tf.keras.layers.Dropout(0.3, name="dropout")(x)   # regularisation

# Single sigmoid output: probability that the image is FAKE
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name="output")(x)

model = tf.keras.Model(inputs, outputs, name="FakeProductClassifier")

# ── Compile ───────────────────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_FROZEN),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

model.summary()

# Print a human-readable parameter summary
total_params     = model.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen  parameters   : {total_params - trainable_params:,}")

## 7. Train Phase 1 — Frozen Base

Train only the classification head for up to `EPOCHS_FROZEN` epochs.
Early stopping will halt training if `val_loss` does not improve for 5 consecutive epochs
and automatically restore the best weights.

In [ ]:
# ── Callbacks for Phase 1 ─────────────────────────────────────────────────────
callbacks1 = [
    # Stop early if validation loss stops improving; restore best weights
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    # Save the best model weights to disk at each epoch
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model_phase1.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    # Reduce learning rate when val_loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,      # halve the learning rate
        patience=3,      # wait 3 epochs before reducing
        min_lr=1e-7,     # lower bound
        verbose=1,
    ),
    # Log metrics for TensorBoard visualisation
    tf.keras.callbacks.TensorBoard(
        log_dir=f'logs/phase1/{datetime.now().strftime("%Y%m%d-%H%M%S")}',
        histogram_freq=1,
    ),
]

print("Starting Phase 1 training (frozen base) …")
print(f"Max epochs: {EPOCHS_FROZEN}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE_FROZEN}")
print("-" * 60)

history1 = model.fit(
    train_ds,
    epochs=EPOCHS_FROZEN,
    validation_data=val_ds,
    callbacks=callbacks1,
)

phase1_epochs = len(history1.history['loss'])
best_val_acc  = max(history1.history['val_accuracy'])
print(f"\nPhase 1 complete: {phase1_epochs} epochs trained.")
print(f"Best validation accuracy: {best_val_acc:.4f}")

## 8. Plot Phase 1 Training Curves

Visualise training vs. validation accuracy and loss.
A large gap between the two curves indicates overfitting — collect more data or increase augmentation.

In [ ]:
def plot_history(history, title_prefix="Phase 1", offset=0, ax_acc=None, ax_loss=None):
    """Plot accuracy and loss curves from a Keras History object.

    Args:
        history:      Keras History object.
        title_prefix: Label to prepend to subplot titles.
        offset:       Epoch offset for x-axis (used when combining histories).
        ax_acc:       Existing Axes for accuracy (None = create new figure).
        ax_loss:      Existing Axes for loss.
    """
    epochs = range(offset + 1, offset + len(history.history['loss']) + 1)

    standalone = ax_acc is None
    if standalone:
        fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"{title_prefix} Training Curves", fontsize=13, fontweight='bold')

    # ── Accuracy subplot ──
    ax_acc.plot(epochs, history.history['accuracy'],     label='Train acc',  linewidth=2)
    ax_acc.plot(epochs, history.history['val_accuracy'], label='Val acc',    linewidth=2, linestyle='--')
    ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_title(f"{title_prefix} — Accuracy")
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)

    # ── Loss subplot ──
    ax_loss.plot(epochs, history.history['loss'],     label='Train loss', linewidth=2)
    ax_loss.plot(epochs, history.history['val_loss'], label='Val loss',   linewidth=2, linestyle='--')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_title(f"{title_prefix} — Loss")
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    if standalone:
        # Mark the early-stopping point (last epoch trained vs. max epochs)
        actual_epochs = len(history.history['loss'])
        if actual_epochs < EPOCHS_FROZEN:
            for ax in (ax_acc, ax_loss):
                ax.axvline(
                    x=offset + actual_epochs,
                    color='red', linestyle=':', linewidth=1.5,
                    label=f'Early stop (epoch {offset + actual_epochs})'
                )
            ax_acc.legend()
            ax_loss.legend()
        plt.tight_layout()
        plt.savefig("phase1_training_curves.png", dpi=100, bbox_inches='tight')
        plt.show()
        print("Figure saved as phase1_training_curves.png")

# Plot Phase 1 curves
plot_history(history1, title_prefix="Phase 1 (Frozen Base)")

## 9. Fine-Tune — Unfreeze Top Layers (Phase 2)

After the head has converged, we unfreeze the last `FINE_TUNE_AT` layers of MobileNetV3Small
and continue training at a much lower learning rate (`1e-5`).
This allows the pre-trained features to adapt to the specific appearance of authentic vs. fake products
without catastrophically forgetting ImageNet features.

In [ ]:
# ── Unfreeze the base model ───────────────────────────────────────────────────
base_model.trainable = True   # allow all layers to be updated ...

# ... but immediately re-freeze every layer EXCEPT the last FINE_TUNE_AT layers
for layer in base_model.layers[:-FINE_TUNE_AT]:
    layer.trainable = False

# Report how many layers are now trainable
trainable_layers = [l for l in base_model.layers if l.trainable]
frozen_layers    = [l for l in base_model.layers if not l.trainable]
print(f"Base model — trainable layers : {len(trainable_layers)}")
print(f"Base model — frozen layers    : {len(frozen_layers)}")
print(f"Fine-tuning the last {FINE_TUNE_AT} layers of '{base_model.name}'.")

# ── Recompile with a lower learning rate ──────────────────────────────────────
# IMPORTANT: always recompile after changing trainability
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_FINETUNE),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

model.summary()

# Updated trainable parameter count
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
total_params     = model.count_params()
print(f"\nTrainable parameters (Phase 2): {trainable_params:,}")
print(f"Frozen  parameters  (Phase 2): {total_params - trainable_params:,}")

## 10. Train Phase 2 — Fine-Tuning

Continue training from where Phase 1 left off.
Using `initial_epoch` ensures the epoch counter in logs is continuous across both phases.

In [ ]:
# ── Callbacks for Phase 2 ─────────────────────────────────────────────────────
callbacks2 = [
    # Stop early and restore best weights
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    # Save the best fine-tuned model
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model_final.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    # Reduce LR if fine-tuning plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,    # even smaller floor for fine-tuning
        verbose=1,
    ),
]

# Phase 2 starts from the last epoch of Phase 1 for continuous logging
initial_epoch = len(history1.history['loss'])
total_epochs  = initial_epoch + EPOCHS_FINETUNE

print("Starting Phase 2 fine-tuning …")
print(f"Continuing from epoch {initial_epoch + 1} → {total_epochs}  |  LR: {LEARNING_RATE_FINETUNE}")
print("-" * 60)

history2 = model.fit(
    train_ds,
    epochs=total_epochs,          # total epoch count (not additional)
    initial_epoch=initial_epoch,  # skip epochs already done in Phase 1
    validation_data=val_ds,
    callbacks=callbacks2,
)

phase2_epochs = len(history2.history['loss'])
best_val_acc2 = max(history2.history['val_accuracy'])
print(f"\nPhase 2 complete: {phase2_epochs} epochs trained.")
print(f"Best validation accuracy (Phase 2): {best_val_acc2:.4f}")

## 11. Plot Combined Training History

Merge the two history objects and plot a single continuous accuracy and loss curve.
A vertical dashed line marks the boundary between Phase 1 and Phase 2.

In [ ]:
# ── Combine both histories ────────────────────────────────────────────────────
def combine_histories(h1, h2):
    """Concatenate metric lists from two Keras History objects."""
    combined = {}
    for key in h1.history:
        if key in h2.history:
            combined[key] = h1.history[key] + h2.history[key]
        else:
            combined[key] = h1.history[key]
    return combined

combined = combine_histories(history1, history2)

# Total number of epochs across both phases
total_epochs_trained = len(combined['loss'])
epoch_range = range(1, total_epochs_trained + 1)

# Epoch at which Phase 2 began (used for the boundary line)
phase_boundary = len(history1.history['loss'])

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Combined Training History (Phase 1 + Phase 2)", fontsize=13, fontweight='bold')

# ── Accuracy ──
ax_acc.plot(epoch_range, combined['accuracy'],     label='Train accuracy', linewidth=2)
ax_acc.plot(epoch_range, combined['val_accuracy'], label='Val accuracy',   linewidth=2, linestyle='--')
ax_acc.axvline(x=phase_boundary, color='orange', linestyle=':', linewidth=2,
               label=f'Fine-tune start (epoch {phase_boundary})')
ax_acc.set_xlabel('Epoch')
ax_acc.set_ylabel('Accuracy')
ax_acc.set_title('Accuracy')
ax_acc.legend()
ax_acc.grid(True, alpha=0.3)

# ── Loss ──
ax_loss.plot(epoch_range, combined['loss'],     label='Train loss', linewidth=2)
ax_loss.plot(epoch_range, combined['val_loss'], label='Val loss',   linewidth=2, linestyle='--')
ax_loss.axvline(x=phase_boundary, color='orange', linestyle=':', linewidth=2,
                label=f'Fine-tune start (epoch {phase_boundary})')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.set_title('Loss')
ax_loss.legend()
ax_loss.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("combined_training_curves.png", dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved as combined_training_curves.png")

## 12. Evaluate on Validation Set

Compute accuracy, precision, recall, and F1 on the full validation set.
The confusion matrix shows the exact breakdown of true/false positives and negatives.

- **Precision** — of all images predicted as *fake*, how many actually are?
- **Recall** — of all real *fake* images, how many did the model catch?
- **F1** — harmonic mean of precision and recall (balanced metric).

In [ ]:
# ── Keras built-in evaluation ─────────────────────────────────────────────────
print("=" * 50)
print("Keras model.evaluate() on validation set:")
print("=" * 50)
eval_results = model.evaluate(val_ds, verbose=1)
eval_names   = model.metrics_names
for name, value in zip(eval_names, eval_results):
    print(f"  {name:12s}: {value:.4f}")

# ── Collect all predictions and true labels ───────────────────────────────────
y_true_list = []   # ground-truth binary labels
y_pred_list = []   # predicted probabilities (0.0 – 1.0)

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)   # shape: (batch, 1)
    y_pred_list.extend(preds.flatten().tolist())
    y_true_list.extend(labels.numpy().flatten().tolist())

y_true  = np.array(y_true_list, dtype=int)
y_prob  = np.array(y_pred_list)
y_pred  = (y_prob >= 0.5).astype(int)    # threshold at 0.5 → binary prediction

# ── sklearn metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)

print(f"\n{'Metric':<12} {'Value':>8}")
print("-" * 22)
print(f"{'Accuracy':<12} {acc:>8.4f}")
print(f"{'Precision':<12} {prec:>8.4f}")
print(f"{'Recall':<12} {rec:>8.4f}")
print(f"{'F1':<12} {f1:>8.4f}")

# ── Classification report ─────────────────────────────────────────────────────
print("\nDetailed classification report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       ylabel='True label', xlabel='Predicted label',
       title='Confusion Matrix')

# Annotate each cell with the raw count
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=14)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=100, bbox_inches='tight')
plt.show()
print("Confusion matrix saved as confusion_matrix.png")

# ── Accuracy warning ──────────────────────────────────────────────────────────
if acc < 0.85:
    print("\n" + "!" * 60)
    print("WARNING: Validation accuracy is below 85%.")
    print("The model may not be ready for production use.")
    print("Recommended actions:")
    print("  1. Add more labelled images (aim for ≥500 per class).")
    print("  2. Check for mislabelled images in the dataset.")
    print("  3. Increase EPOCHS_FROZEN or EPOCHS_FINETUNE.")
    print("  4. Try stronger augmentation (e.g., RandomContrast).")
    print("!" * 60)
else:
    print(f"\nAccuracy {acc:.2%} meets the 85% threshold. Model looks good!")

## 13. Convert to TFLite with INT8 Quantization

INT8 quantization reduces the model size by ~4× and speeds up inference on Android devices
that have INT8-capable neural network accelerators (NNAPI, Hexagon DSP).

A **representative dataset** is required for INT8 calibration — it tells the converter
the typical range of activation values so it can choose good quantization scales.
We fall back to float16 quantization if INT8 conversion fails.

In [ ]:
TFLITE_PATH    = Path("product_classifier.tflite")
TFLITE_FP16_PATH = Path("product_classifier_fp16.tflite")

def representative_dataset():
    """Yield batches of float32 images for INT8 calibration.

    TFLite uses these samples to determine the quantization scale and
    zero-point for each layer's activations.
    """
    for images, _ in train_ds.take(100):   # 100 batches ≈ 3200 images
        yield [tf.cast(images, tf.float32)]

# ── Attempt INT8 full-integer quantization ────────────────────────────────────
try:
    print("Converting to TFLite (INT8 full-integer quantization) …")

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]   # enable quantization
    converter.representative_dataset = representative_dataset

    # Restrict to INT8 ops only — required for NNAPI acceleration on Android
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

    # Quantize input and output tensors to int8 as well
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    TFLITE_PATH.write_bytes(tflite_model)   # save to disk
    size_kb = TFLITE_PATH.stat().st_size / 1024
    print(f"INT8 TFLite model saved : {TFLITE_PATH}  ({size_kb:.1f} KB)")

except Exception as e:
    # INT8 can fail if the model contains unsupported ops; fall back to float16
    print(f"INT8 conversion failed: {e}")
    print("Falling back to float16 quantization …")

    try:
        converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
        converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
        # float16 is supported on all TFLite runtimes
        converter_fp16.target_spec.supported_types = [tf.float16]

        tflite_fp16 = converter_fp16.convert()
        TFLITE_FP16_PATH.write_bytes(tflite_fp16)
        size_kb = TFLITE_FP16_PATH.stat().st_size / 1024
        TFLITE_PATH = TFLITE_FP16_PATH   # point to the fallback file
        print(f"float16 TFLite model saved: {TFLITE_FP16_PATH}  ({size_kb:.1f} KB)")
        print("Note: float16 model will NOT use INT8 hardware acceleration.")

    except Exception as e2:
        print(f"float16 conversion also failed: {e2}")
        print("ERROR: Could not export a TFLite model. Check model compatibility.")
        raise

## 14. Test the TFLite Model on Sample Images

Run inference using the exported `.tflite` file to verify the conversion was lossless.
This mimics what the Android app will do at runtime using the TFLite interpreter.

In [ ]:
# ── Load the TFLite model ─────────────────────────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()   # allocate memory for input/output buffers

# Inspect input and output tensor metadata
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Input  details:", input_details[0])
print("TFLite Output details:", output_details[0])

input_dtype  = input_details[0]['dtype']   # np.int8 or np.float32
input_scale, input_zero_point   = input_details[0].get('quantization', (1.0, 0))
output_scale, output_zero_point = output_details[0].get('quantization', (1.0, 0))

print(f"\nInput  dtype : {input_dtype}  (scale={input_scale}, zp={input_zero_point})")
print(f"Output dtype : {output_details[0]['dtype']}  (scale={output_scale}, zp={output_zero_point})")

# ── Collect 5 sample images from the validation set ───────────────────────────
# We iterate the *raw* (non-preprocessed) dataset because the TFLite model
# already includes preprocessing in its graph (via the Lambda layer).
sample_imgs, sample_labels_raw = [], []
for imgs, lbls in val_ds_raw.unbatch().take(5):
    sample_imgs.append(imgs.numpy())             # uint8 HxWxC
    sample_labels_raw.append(int(lbls.numpy()))

print(f"\nRunning TFLite inference on {len(sample_imgs)} sample images …\n")

correct = 0
for idx, (img, true_label) in enumerate(zip(sample_imgs, sample_labels_raw)):
    # Add batch dimension: (H, W, C) → (1, H, W, C)
    img_batch = np.expand_dims(img, axis=0).astype(np.float32)

    # Quantize float input to int8 if the model expects int8
    if input_dtype == np.int8:
        img_batch = (img_batch / input_scale + input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details[0]['index'], img_batch)
    interpreter.invoke()   # run inference

    raw_output = interpreter.get_tensor(output_details[0]['index'])[0][0]

    # Dequantize int8 output back to probability
    if output_details[0]['dtype'] == np.int8:
        prob = (float(raw_output) - output_zero_point) * output_scale
    else:
        prob = float(raw_output)

    pred_label = 1 if prob >= 0.5 else 0
    pred_name  = CLASS_NAMES[pred_label]
    true_name  = CLASS_NAMES[true_label]
    status     = "PASS" if pred_label == true_label else "FAIL"
    correct   += (pred_label == true_label)

    print(
        f"  Sample {idx + 1}: "
        f"true={true_name:10s}  "
        f"pred={pred_name:10s}  "
        f"confidence={prob:.3f}  "
        f"[{status}]"
    )

tflite_accuracy = correct / len(sample_imgs)
print(f"\nTFLite accuracy on {len(sample_imgs)} samples: {tflite_accuracy:.0%}  ({correct}/{len(sample_imgs)} correct)")

## 15. Copy Model to Android App Assets

The Android app reads the `.tflite` model from `app/src/main/assets/`.
This cell copies the exported model there automatically if the directory exists.
If not (e.g., running in Colab), it prints the manual copy command.

In [ ]:
# Path to the Android app assets directory (relative to the scripts/ folder)
ASSETS_DIR = Path("../app/src/main/assets")

if ASSETS_DIR.exists():
    dest = ASSETS_DIR / "product_classifier.tflite"
    shutil.copy(TFLITE_PATH, dest)
    copied_size_kb = dest.stat().st_size / 1024
    print(f"Copied to {dest.resolve()}")
    print(f"Model size in assets: {copied_size_kb:.1f} KB")
else:
    print(f"Assets directory not found at {ASSETS_DIR.resolve()}")
    print("Manual step: copy the TFLite model to the Android assets folder.")
    print()
    print("Run this command from the project root:")
    print(f"  cp {TFLITE_PATH.resolve()} <project-root>/app/src/main/assets/product_classifier.tflite")

# ── Final summary ─────────────────────────────────────────────────────────────
tflite_size_kb = TFLITE_PATH.stat().st_size / 1024

print()
print("=" * 55)
print("  TRAINING COMPLETE — FINAL SUMMARY")
print("=" * 55)
print(f"  TFLite model path : {TFLITE_PATH}")
print(f"  Model size        : {tflite_size_kb:.1f} KB")
print(f"  Val accuracy      : {acc:.2%}")
print(f"  Val precision     : {prec:.2%}")
print(f"  Val recall        : {rec:.2%}")
print(f"  Val F1            : {f1:.2%}")
print(f"  Phase 1 epochs    : {len(history1.history['loss'])}")
print(f"  Phase 2 epochs    : {len(history2.history['loss'])}")
print("=" * 55)

## Summary

### Completed steps

- [x] **Environment setup** — TensorFlow installed, GPU checked.
- [x] **Dataset loaded** — images read from `dataset/train/` and `dataset/validation/`.
- [x] **Data augmentation** — random flip, rotation, zoom, brightness applied to training set.
- [x] **Phase 1 training** — MobileNetV3Small head trained with frozen base.
- [x] **Phase 2 fine-tuning** — top 30 layers unfrozen and trained at low LR.
- [x] **Evaluation** — accuracy, precision, recall, F1, and confusion matrix computed.
- [x] **TFLite export** — INT8-quantized model saved as `product_classifier.tflite`.
- [x] **TFLite test** — inference verified on 5 sample images.
- [x] **Android copy** — model copied to `app/src/main/assets/` (if directory exists).

### Next steps — Android integration

1. **Add TFLite dependency** in `app/build.gradle`:
   ```groovy
   implementation 'org.tensorflow:tensorflow-lite:2.14.0'
   implementation 'org.tensorflow:tensorflow-lite-support:0.4.4'
   ```
2. **Load the model** in your `ProductClassifier.kt`:
   ```kotlin
   val model = ProductClassifier.newInstance(context)
   ```
3. **Preprocess the input image** to 224×224, normalise pixels to `[-1, 1]` (or pass raw uint8 if using the INT8 model with the quantization parameters above).
4. **Run inference** and threshold the output at 0.5:
   - Output ≥ 0.5 → **FAKE** product.
   - Output < 0.5 → **AUTHENTIC** product.
5. **Display the result** to the user with the confidence score.
6. **Re-train periodically** as you collect more labelled product images to keep accuracy high.

> **Tip:** If accuracy is below 85%, the most impactful fix is always to collect more diverse, correctly labelled images — aim for at least 500 images per class.